# Notebook 5: Model Comparison - XGBoost vs Neural Network

**Site Revenue Prediction ML System - SageMaker Notebook Series**

This notebook covers:
- **Part 1: XGBoost for Revenue Prediction** (Regression)
- **Part 2: Neural Network for Top Performer Classification** (Binary Classification)
- Model comparison and evaluation
- Feature importance analysis for both models

## SageMaker Notes
- XGBoost is available as a built-in SageMaker algorithm
- For GPU training, use `ml.p3.2xlarge` instances
- Both models can be deployed as SageMaker endpoints

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import pickle
import time
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print(f"XGBoost version: {xgb.__version__}")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not installed. Install with: pip install xgboost")

plt.style.use('seaborn-v0_8-whitegrid')
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 1. Load Processed Data

In [ ]:
OUTPUT_DIR = Path('./outputs')

# Device selection
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Using device: {device}")

# Load model configuration
with open(OUTPUT_DIR / 'model_config.json', 'r') as f:
    config = json.load(f)

print("\nModel Configuration:")
for key, value in config.items():
    if key != 'categorical_vocab_sizes':
        print(f"  {key}: {value}")

In [ ]:
# Load processed data
data = torch.load(OUTPUT_DIR / 'processed_data.pt', weights_only=False)

numeric = data['numeric']
categorical = data['categorical']
boolean = data['boolean']
target = data['target']
train_idx = data['train_idx']
val_idx = data['val_idx']
test_idx = data['test_idx']

print(f"Dataset sizes:")
print(f"  Train: {len(train_idx):,}")
print(f"  Val:   {len(val_idx):,}")
print(f"  Test:  {len(test_idx):,}")
print(f"\nFeature shapes:")
print(f"  Numeric:     {numeric.shape}")
print(f"  Categorical: {categorical.shape}")
print(f"  Boolean:     {boolean.shape}")

In [ ]:
# Load preprocessor for inverse transform
with open(OUTPUT_DIR / 'preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

target_scaler = preprocessor['target_scaler']
feature_names = (
    preprocessor.get('available_numeric', []) +
    list(config['categorical_vocab_sizes'].keys()) +
    preprocessor.get('available_boolean', [])
)
print(f"Total features: {len(feature_names)}")

In [ ]:
# Prepare data for XGBoost (concatenate all features)
X_all = np.concatenate([
    numeric.numpy(),
    categorical.numpy().astype(np.float32),
    boolean.numpy()
], axis=1)

# Get target in original scale
if target_scaler is not None:
    y_all = target_scaler.inverse_transform(target.numpy()).flatten()
else:
    y_all = target.numpy().flatten()

# Split data
X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]

print(f"X_train shape: {X_train.shape}")
print(f"y_train range: ${y_train.min():.0f} - ${y_train.max():.0f}")

---

# Part 1: XGBoost for Revenue Prediction (Regression)

XGBoost (eXtreme Gradient Boosting) is a powerful gradient boosting algorithm that:
- Handles mixed feature types naturally
- Provides built-in feature importance
- Often achieves state-of-the-art results on tabular data
- Is highly optimized for speed and memory efficiency

In [ ]:
if XGBOOST_AVAILABLE:
    # XGBoost hyperparameters for regression
    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'max_depth': 8,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 5,
        'reg_alpha': 0.1,  # L1 regularization
        'reg_lambda': 1.0,  # L2 regularization
        'n_estimators': 500,
        'early_stopping_rounds': 50,
        'random_state': 42,
        'n_jobs': -1,
    }
    
    # Use GPU if available
    if torch.cuda.is_available():
        xgb_params['tree_method'] = 'gpu_hist'
        xgb_params['device'] = 'cuda'
    
    print("XGBoost Parameters:")
    for k, v in xgb_params.items():
        print(f"  {k}: {v}")
else:
    print("Skipping XGBoost - not installed")

In [ ]:
if XGBOOST_AVAILABLE:
    print("Training XGBoost Regression Model...")
    start_time = time.time()
    
    # Create XGBoost regressor
    xgb_model = xgb.XGBRegressor(**xgb_params)
    
    # Train with early stopping
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50
    )
    
    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time:.1f} seconds")
    print(f"Best iteration: {xgb_model.best_iteration}")

In [ ]:
if XGBOOST_AVAILABLE:
    # Make predictions
    y_pred_xgb = xgb_model.predict(X_test)
    
    # Calculate metrics
    mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
    rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    r2_xgb = r2_score(y_test, y_pred_xgb)
    
    # SMAPE
    smape_xgb = np.mean(2 * np.abs(y_test - y_pred_xgb) / (np.abs(y_test) + np.abs(y_pred_xgb) + 1e-8)) * 100
    
    print("XGBoost Regression - Test Set Results:")
    print("=" * 50)
    print(f"  MAE:   ${mae_xgb:,.2f}")
    print(f"  RMSE:  ${rmse_xgb:,.2f}")
    print(f"  R2:    {r2_xgb:.4f}")
    print(f"  SMAPE: {smape_xgb:.2f}%")

In [ ]:
if XGBOOST_AVAILABLE:
    # Feature importance
    importance = xgb_model.feature_importances_
    
    # Create feature importance dataframe
    if len(feature_names) == len(importance):
        feat_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': importance
        }).sort_values('importance', ascending=False)
    else:
        feat_importance = pd.DataFrame({
            'feature': [f'feature_{i}' for i in range(len(importance))],
            'importance': importance
        }).sort_values('importance', ascending=False)
    
    print("\nTop 15 Most Important Features (XGBoost):")
    print("=" * 50)
    for i, (_, row) in enumerate(feat_importance.head(15).iterrows()):
        bar = '█' * int(row['importance'] / feat_importance['importance'].max() * 20)
        print(f"{i+1:2}. {row['feature']:35} {row['importance']:.4f} {bar}")

In [ ]:
if XGBOOST_AVAILABLE:
    # Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Actual vs Predicted
    ax = axes[0]
    ax.scatter(y_test, y_pred_xgb, alpha=0.3, s=10)
    max_val = max(y_test.max(), y_pred_xgb.max())
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    ax.set_xlabel('Actual Revenue ($)')
    ax.set_ylabel('Predicted Revenue ($)')
    ax.set_title(f'XGBoost: Actual vs Predicted (R² = {r2_xgb:.4f})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Feature Importance
    ax = axes[1]
    top_features = feat_importance.head(15)
    ax.barh(range(len(top_features)), top_features['importance'].values[::-1])
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels(top_features['feature'].values[::-1])
    ax.set_xlabel('Feature Importance')
    ax.set_title('XGBoost: Top 15 Feature Importances')
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'xgboost_regression_results.png', dpi=150)
    plt.show()

---

# Part 2: Neural Network for Binary Classification (Lookalike Model)

The Lookalike model identifies sites that are similar to top performers:
- **Positive class**: Top 10% revenue sites ("top performers")
- **Negative class**: Bottom 90% sites
- Uses the same neural network architecture with sigmoid output
- Optimized with Binary Cross-Entropy loss

In [ ]:
# Create binary classification target (top 10% performers)
threshold_90 = np.percentile(y_all, 90)
y_binary_all = (y_all >= threshold_90).astype(np.float32)

# Split binary target
y_train_binary = y_binary_all[train_idx]
y_val_binary = y_binary_all[val_idx]
y_test_binary = y_binary_all[test_idx]

print(f"Classification Target (Lookalike Model):")
print("=" * 50)
print(f"Threshold (90th percentile): ${threshold_90:,.2f}")
print(f"\nTrain: {int(y_train_binary.sum()):,} positive ({y_train_binary.mean()*100:.1f}%)")
print(f"Val:   {int(y_val_binary.sum()):,} positive ({y_val_binary.mean()*100:.1f}%)")
print(f"Test:  {int(y_test_binary.sum()):,} positive ({y_test_binary.mean()*100:.1f}%)")

In [ ]:
# Neural Network Architecture for Classification
class ResidualBlock(nn.Module):
    """Residual block with batch normalization."""
    def __init__(self, in_features, out_features, dropout=0.2):
        super().__init__()
        self.linear1 = nn.Linear(in_features, out_features)
        self.bn1 = nn.BatchNorm1d(out_features)
        self.linear2 = nn.Linear(out_features, out_features)
        self.bn2 = nn.BatchNorm1d(out_features)
        self.dropout = nn.Dropout(dropout)
        self.projection = nn.Linear(in_features, out_features) if in_features != out_features else nn.Identity()
    
    def forward(self, x):
        residual = self.projection(x)
        out = torch.relu(self.bn1(self.linear1(x)))
        out = self.dropout(out)
        out = self.bn2(self.linear2(out))
        return torch.relu(out + residual)


class CategoricalEmbedding(nn.Module):
    """Embedding layer for categorical features."""
    def __init__(self, vocab_sizes, embedding_dim=16):
        super().__init__()
        self.embeddings = nn.ModuleDict()
        self.feature_names = list(vocab_sizes.keys())
        for name, vocab_size in vocab_sizes.items():
            dim = min(embedding_dim, max(4, (vocab_size + 1) // 2))
            self.embeddings[name] = nn.Embedding(vocab_size + 1, dim, padding_idx=0)
        self.output_dim = sum(self.embeddings[name].embedding_dim for name in self.feature_names)
    
    def forward(self, x):
        embeddings = []
        for i, name in enumerate(self.feature_names):
            idx = x[:, i].clamp(0, self.embeddings[name].num_embeddings - 1)
            embeddings.append(self.embeddings[name](idx))
        return torch.cat(embeddings, dim=1)


class LookalikeClassifier(nn.Module):
    """
    Neural network for binary classification (Lookalike Model).
    Predicts probability of a site being a "top performer".
    """
    def __init__(self, n_numeric, n_boolean, categorical_vocab_sizes,
                 embedding_dim=16, hidden_dims=None, dropout=0.3):
        super().__init__()
        hidden_dims = hidden_dims or [256, 128, 64]
        
        self.cat_embedding = CategoricalEmbedding(categorical_vocab_sizes, embedding_dim)
        self.numeric_bn = nn.BatchNorm1d(n_numeric) if n_numeric > 0 else None
        self.n_numeric = n_numeric
        self.n_boolean = n_boolean
        
        total_input_dim = self.cat_embedding.output_dim + n_numeric + n_boolean
        
        # Build MLP
        layers = []
        in_dim = total_input_dim
        for out_dim in hidden_dims:
            layers.append(ResidualBlock(in_dim, out_dim, dropout))
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        
        # Output layer (single neuron for binary classification)
        self.output = nn.Linear(hidden_dims[-1], 1)
        
        self._init_weights()
    
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight, gain=0.5)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
    
    def forward(self, numeric, categorical, boolean):
        cat_embedded = self.cat_embedding(categorical)
        if self.numeric_bn is not None and self.n_numeric > 0:
            numeric = self.numeric_bn(numeric)
        x = torch.cat([cat_embedded, numeric, boolean], dim=1)
        x = self.mlp(x)
        return self.output(x)  # Raw logits
    
    def predict_proba(self, numeric, categorical, boolean):
        """Return probability of positive class."""
        logits = self.forward(numeric, categorical, boolean)
        return torch.sigmoid(logits)

print("LookalikeClassifier architecture defined")

In [ ]:
# Create PyTorch Dataset for classification
class ClassificationDataset(Dataset):
    def __init__(self, numeric, categorical, boolean, target):
        self.numeric = numeric
        self.categorical = categorical
        self.boolean = boolean
        self.target = target
    
    def __len__(self):
        return len(self.target)
    
    def __getitem__(self, idx):
        return self.numeric[idx], self.categorical[idx], self.boolean[idx], self.target[idx]

# Create datasets with binary targets
train_dataset_clf = ClassificationDataset(
    numeric[train_idx], categorical[train_idx], boolean[train_idx],
    torch.tensor(y_train_binary).unsqueeze(1)
)
val_dataset_clf = ClassificationDataset(
    numeric[val_idx], categorical[val_idx], boolean[val_idx],
    torch.tensor(y_val_binary).unsqueeze(1)
)
test_dataset_clf = ClassificationDataset(
    numeric[test_idx], categorical[test_idx], boolean[test_idx],
    torch.tensor(y_test_binary).unsqueeze(1)
)

# DataLoaders
BATCH_SIZE = 2048
import platform
NUM_WORKERS = 0 if platform.system() == 'Darwin' else 4

train_loader_clf = DataLoader(train_dataset_clf, batch_size=BATCH_SIZE, shuffle=True, 
                               num_workers=NUM_WORKERS, drop_last=True)
val_loader_clf = DataLoader(val_dataset_clf, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS)
test_loader_clf = DataLoader(test_dataset_clf, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS)

print(f"Train batches: {len(train_loader_clf)}, Val batches: {len(val_loader_clf)}")

In [ ]:
# Initialize classification model
clf_model = LookalikeClassifier(
    n_numeric=config['n_numeric'],
    n_boolean=config['n_boolean'],
    categorical_vocab_sizes=config['categorical_vocab_sizes'],
    embedding_dim=16,
    hidden_dims=[256, 128, 64],
    dropout=0.3,
)
clf_model = clf_model.to(device)

# Calculate class weights for imbalanced data
n_positive = y_train_binary.sum()
n_negative = len(y_train_binary) - n_positive
pos_weight = torch.tensor([n_negative / n_positive]).to(device)

# Loss function with class weighting
criterion_clf = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer
optimizer_clf = AdamW(clf_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_clf = ReduceLROnPlateau(optimizer_clf, mode='max', factor=0.5, patience=5)

print(f"Model parameters: {sum(p.numel() for p in clf_model.parameters()):,}")
print(f"Class weight (pos_weight): {pos_weight.item():.2f}")

In [ ]:
# Training functions for classification
def train_epoch_clf(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for numeric, categorical, boolean, target in train_loader:
        numeric = numeric.to(device)
        categorical = categorical.to(device)
        boolean = boolean.to(device)
        target = target.to(device)
        
        optimizer.zero_grad()
        logits = model(numeric, categorical, boolean)
        loss = criterion(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate_clf(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_targets = []
    
    for numeric, categorical, boolean, target in val_loader:
        numeric = numeric.to(device)
        categorical = categorical.to(device)
        boolean = boolean.to(device)
        target = target.to(device)
        
        logits = model(numeric, categorical, boolean)
        loss = criterion(logits, target)
        total_loss += loss.item()
        
        probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu())
        all_targets.append(target.cpu())
    
    probs = torch.cat(all_probs).numpy().flatten()
    targets = torch.cat(all_targets).numpy().flatten()
    
    # Calculate AUC
    try:
        auc = roc_auc_score(targets, probs)
    except:
        auc = 0.5
    
    # Calculate accuracy at 0.5 threshold
    preds = (probs >= 0.5).astype(int)
    acc = accuracy_score(targets, preds)
    
    return total_loss / len(val_loader), auc, acc

print("Training functions defined")

In [ ]:
# Train classification model
EPOCHS = 50
best_auc = 0.0
patience_counter = 0
PATIENCE = 10
history_clf = []
best_model_state = None

print(f"Training Lookalike Classifier for {EPOCHS} epochs...")
print("=" * 70)

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch_clf(clf_model, train_loader_clf, criterion_clf, optimizer_clf, device)
    val_loss, val_auc, val_acc = evaluate_clf(clf_model, val_loader_clf, criterion_clf, device)
    
    scheduler_clf.step(val_auc)
    current_lr = optimizer_clf.param_groups[0]['lr']
    
    history_clf.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_auc': val_auc,
        'val_acc': val_acc,
    })
    
    print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"AUC: {val_auc:.4f} | Acc: {val_acc:.4f} | LR: {current_lr:.2e}")
    
    if val_auc > best_auc:
        best_auc = val_auc
        patience_counter = 0
        best_model_state = clf_model.state_dict().copy()
        print(f"  → New best model! (AUC: {val_auc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

# Restore best model
if best_model_state is not None:
    clf_model.load_state_dict(best_model_state)

total_time = time.time() - start_time
print("=" * 70)
print(f"Training completed in {total_time:.1f} seconds")
print(f"Best validation AUC: {best_auc:.4f}")

In [ ]:
# Evaluate on test set
clf_model.eval()
all_probs = []
all_targets = []

with torch.no_grad():
    for numeric_batch, cat_batch, bool_batch, target_batch in test_loader_clf:
        numeric_batch = numeric_batch.to(device)
        cat_batch = cat_batch.to(device)
        bool_batch = bool_batch.to(device)
        
        probs = clf_model.predict_proba(numeric_batch, cat_batch, bool_batch)
        all_probs.append(probs.cpu())
        all_targets.append(target_batch)

test_probs = torch.cat(all_probs).numpy().flatten()
test_targets = torch.cat(all_targets).numpy().flatten()
test_preds = (test_probs >= 0.5).astype(int)

# Metrics
test_auc = roc_auc_score(test_targets, test_probs)
test_acc = accuracy_score(test_targets, test_preds)
test_precision = precision_score(test_targets, test_preds, zero_division=0)
test_recall = recall_score(test_targets, test_preds, zero_division=0)
test_f1 = f1_score(test_targets, test_preds, zero_division=0)

print("Neural Network Classification - Test Set Results:")
print("=" * 50)
print(f"  AUC-ROC:   {test_auc:.4f}")
print(f"  Accuracy:  {test_acc:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1 Score:  {test_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(test_targets, test_preds, target_names=['Non-Top', 'Top Performer']))

In [ ]:
# Visualizations for classification
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Training curves
ax = axes[0, 0]
epochs = [h['epoch'] for h in history_clf]
ax.plot(epochs, [h['train_loss'] for h in history_clf], label='Train Loss')
ax.plot(epochs, [h['val_loss'] for h in history_clf], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# AUC curve
ax = axes[0, 1]
ax.plot(epochs, [h['val_auc'] for h in history_clf], color='green')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC-ROC')
ax.set_title('Validation AUC-ROC')
ax.grid(True, alpha=0.3)

# ROC Curve
ax = axes[1, 0]
fpr, tpr, thresholds = roc_curve(test_targets, test_probs)
ax.plot(fpr, tpr, color='blue', linewidth=2, label=f'ROC (AUC = {test_auc:.4f})')
ax.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()
ax.grid(True, alpha=0.3)

# Confusion Matrix
ax = axes[1, 1]
cm = confusion_matrix(test_targets, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-Top', 'Top Performer'],
            yticklabels=['Non-Top', 'Top Performer'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nn_classification_results.png', dpi=150)
plt.show()

---

# Part 3: Model Comparison Summary

In [ ]:
# Summary comparison
print("="* 70)
print("MODEL COMPARISON SUMMARY")
print("="* 70)

print("\n1. XGBoost Regression (Revenue Prediction):")
print("-" * 50)
if XGBOOST_AVAILABLE:
    print(f"   MAE:   ${mae_xgb:,.2f}")
    print(f"   RMSE:  ${rmse_xgb:,.2f}")
    print(f"   R²:    {r2_xgb:.4f}")
    print(f"   SMAPE: {smape_xgb:.2f}%")
else:
    print("   [XGBoost not available]")

print("\n2. Neural Network Classification (Lookalike Model):")
print("-" * 50)
print(f"   AUC-ROC:   {test_auc:.4f}")
print(f"   Accuracy:  {test_acc:.4f}")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall:    {test_recall:.4f}")
print(f"   F1 Score:  {test_f1:.4f}")

print("\n" + "="* 70)
print("RECOMMENDATIONS:")
print("="* 70)
print("""
• Use XGBoost Regression for:
  - Direct revenue predictions
  - Understanding feature importance
  - Quick training and inference

• Use Neural Network Classification for:
  - Identifying potential high performers
  - Lead scoring and prioritization
  - Sites similar to top performers ("lookalikes")
""")

In [ ]:
# Save models
if XGBOOST_AVAILABLE:
    xgb_model.save_model(OUTPUT_DIR / 'xgboost_regression.json')
    print(f"XGBoost model saved to {OUTPUT_DIR / 'xgboost_regression.json'}")

# Save classification model
torch.save({
    'model_state_dict': clf_model.state_dict(),
    'config': {
        'n_numeric': config['n_numeric'],
        'n_boolean': config['n_boolean'],
        'categorical_vocab_sizes': config['categorical_vocab_sizes'],
        'hidden_dims': [256, 128, 64],
        'dropout': 0.3,
    },
    'threshold': threshold_90,
    'best_auc': best_auc,
    'test_metrics': {
        'auc': test_auc,
        'accuracy': test_acc,
        'precision': test_precision,
        'recall': test_recall,
        'f1': test_f1,
    }
}, OUTPUT_DIR / 'lookalike_classifier.pt')
print(f"Lookalike classifier saved to {OUTPUT_DIR / 'lookalike_classifier.pt'}")

In [ ]:
# Save comparison summary
comparison_summary = {
    'xgboost_regression': {
        'available': XGBOOST_AVAILABLE,
        'mae': float(mae_xgb) if XGBOOST_AVAILABLE else None,
        'rmse': float(rmse_xgb) if XGBOOST_AVAILABLE else None,
        'r2': float(r2_xgb) if XGBOOST_AVAILABLE else None,
        'smape': float(smape_xgb) if XGBOOST_AVAILABLE else None,
    },
    'nn_classification': {
        'threshold_90th_percentile': float(threshold_90),
        'auc': float(test_auc),
        'accuracy': float(test_acc),
        'precision': float(test_precision),
        'recall': float(test_recall),
        'f1': float(test_f1),
    }
}

with open(OUTPUT_DIR / 'model_comparison.json', 'w') as f:
    json.dump(comparison_summary, f, indent=2)

print(f"Comparison summary saved to {OUTPUT_DIR / 'model_comparison.json'}")

---

## Summary

This notebook has:
1. ✅ Trained XGBoost for revenue prediction (regression)
2. ✅ Analyzed XGBoost feature importance
3. ✅ Built Neural Network for binary classification (Lookalike model)
4. ✅ Evaluated both models with comprehensive metrics
5. ✅ Created visualizations for both models
6. ✅ Saved trained models and comparison summary

**Artifacts created:**
- `outputs/xgboost_regression.json` - Trained XGBoost model
- `outputs/lookalike_classifier.pt` - Trained Neural Network classifier
- `outputs/xgboost_regression_results.png` - XGBoost visualizations
- `outputs/nn_classification_results.png` - Classification visualizations
- `outputs/model_comparison.json` - Comparison metrics

**Next:** Proceed to `06_explainability.ipynb` for model interpretability (SHAP, calibration, tiers).